#Python OCR model to read out: Backend will be 90% python

In [15]:
%pip install pandas
%pip install matplotlib
%pip install plotly


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
%pip install pytesseract


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
import pandas as pd
import pytesseract
from PIL import Image

print(pytesseract.get_tesseract_version())

5.3.4


# Extract_text functions: parses the texts from the images to then be used for the model

In [19]:
from PIL import ImageOps

def extract_text(image_path):
    img = Image.open(image_path)
    img = ImageOps.grayscale(img)                              # simpler colors = better accuracy
    img = img.resize((img.width * 2, img.height * 2))          # upscaling helps with small screenshot text
    text = pytesseract.image_to_string(img)
    return text.strip()

In [20]:
from PIL import ImageDraw, ImageFont
import os, re, textwrap

# --- Metrics ---
def levenshtein(a, b):
    """Minimum number of edits to turn sequence a into sequence b."""
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        curr = [i]
        for j, y in enumerate(b, 1):
            curr.append(min(prev[j] + 1,              # deletion
                            curr[j - 1] + 1,          # insertion
                            prev[j - 1] + (x != y)))  # substitution
        prev = curr
    return prev[-1]

def normalize(text):
    return re.sub(r"\s+", " ", str(text)).strip().lower()

def cer(truth, ocr):
    truth, ocr = normalize(truth), normalize(ocr)
    return levenshtein(truth, ocr) / max(len(truth), 1)

def wer(truth, ocr):
    t, o = normalize(truth).split(), normalize(ocr).split()
    return levenshtein(t, o) / max(len(t), 1)

In [26]:
import glob

# Show every CSV in the project (searching from the folder above Backend)
for f in glob.glob("../**/*.csv", recursive=True):
    print(f)

                      # the exact text in the image = ground truth

../data/done/phishing_cleaned.csv
../data/done/legit_cleaned.csv
../data/done/clean_phishing_email2.csv
../data/done/assassin_phishing_cleaned.csv
../data/done/assassin_legit_cleaned.csv
../data/done/trec06_legit_cleaned.csv
../data/done/real_phishing_email2.csv
../data/done/trec06_phishing_cleaned.csv
../data/not_done/phishing_emailDetectionv2.csv
../data/not_done/Phishing_Email.csv
../data/not_done/Assassin.csv
../data/not_done/TREC-06.csv


In [27]:
import glob, os, re
import pandas as pd

SCREENSHOT_DIR = "../rendered_email_screenshots"   # notebook runs from Backend/, so go up one level

# Sort numerically so email_2 comes before email_10
def natural_key(path):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", path)]

image_paths = sorted(
    glob.glob(os.path.join(SCREENSHOT_DIR, "**", "*.png"), recursive=True),
    key=natural_key,
)
print(f"Found {len(image_paths)} images")

results = []
for i, path in enumerate(image_paths, 1):
    folder = os.path.basename(os.path.dirname(path))      # e.g. "trec06_phishing"
    try:
        text = extract_text(path)
        error = None
    except Exception as e:                                # one bad image won't stop the loop
        text, error = "", str(e)

    results.append({
        "file": os.path.basename(path),
        "folder": folder,
        "label": 1 if "phishing" in folder else 0,        # 1 = phishing, 0 = legit
        "text": text,
        "error": error,
    })

    if i % 25 == 0 or i == len(image_paths):
        print(f"{i}/{len(image_paths)} images read")

ocr_df = pd.DataFrame(results)
ocr_df.to_csv("../data/ocr_results.csv", index=False)
print("Saved to data/ocr_results.csv")
ocr_df.head()

Found 200 images
25/200 images read
50/200 images read
75/200 images read
100/200 images read
125/200 images read
150/200 images read
175/200 images read
200/200 images read
Saved to data/ocr_results.csv


,file,folder,label,text,error
0,assassin_legit_email_0.png,assassin_legit,0,Re: New Sequences Window\n\nFrom: Robert Elz <...,None
1,assassin_legit_email_1.png,assassin_legit,0,[zzzzteana] RE: Alexander\n\nFrom: Steve Burt ...,None
2,assassin_legit_email_2.png,assassin_legit,0,"[zzzzteana] Moscow bomber\n\nFrom: ""Tim Chapma...",None
3,assassin_legit_email_3.png,assassin_legit,0,[IRR] Klez: The Virus That Won't Die\n\nFrom: ...,None
4,assassin_legit_email_4.png,assassin_legit,0,Re: [zzzzteana] Nothing like mama used to make...,None


In [28]:
empty = ocr_df["text"].str.len() < 20
print("Images read:", len(ocr_df))
print("Errors:", ocr_df["error"].notna().sum())
print("Empty/near-empty results:", empty.sum())
print(ocr_df.groupby("folder").size())

Images read: 200
Errors: 0
Empty/near-empty results: 0
folder
assassin_legit       50
assassin_phishing    50
trec06_legit         50
trec06_phishing      50
dtype: int64


In [38]:
#first_52 = ocr_df.head(52)      # or: ocr_df.iloc[:51]
#first_52

pd.set_option("display.max_rows", 60)          # show all 51 rows instead of "..."
pd.set_option("display.max_colwidth", 200)     # show more of each email's text

ocr_df.head(51)

,file,folder,label,text,error
0,assassin_legit_email_0.png,assassin_legit,0,"Re: New Sequences Window\n\nFrom: Robert Elz <kre@munnari.OZ.AU>\n\nDate: Wed, 21 Aug 2002 10:54:46 -0500_ From: Chris Garrigues Message-ID:\n<1029945287.4797.TMDA@deepeddy.vircio.com> || can't re...",None
1,assassin_legit_email_1.png,assassin_legit,0,"[zzzzteana] RE: Alexander\n\nFrom: Steve Burt <Steve_Burt@cursor-system.com>\n\nMartin A posted:\n\nTassos Papadopoulos, the Greek sculptor behind the plan, judged that the\nlimestone of Mount Ker...",None
2,assassin_legit_email_2.png,assassin_legit,0,"[zzzzteana] Moscow bomber\n\nFrom: ""Tim Chapman"" <timc@2ubh.com>\n\nMan Threatens Explosion In Moscow\n\nThursday August 22, 2002 1:40 PM\n\nMOSCOW (AP) - Security officers on Thursday seized an u...",None
3,assassin_legit_email_3.png,assassin_legit,0,"[IRR] Klez: The Virus That Won't Die\n\nFrom: Monty Solomon <monty@roscom.com>\n\nKlez: The Virus That Won't Die\nAlready the most prolific virus ever, Klez continues to wreak havoc.\n\nAndrew Bra...",None
4,assassin_legit_email_4.png,assassin_legit,0,"Re: [zzzzteana] Nothing like mama used to make\n\nFrom: Stewart Smith <Stewart.Smith@ee.ed.ac.uk>\n\n> in adding cream to spaghetti carbonara, which has the same effect on pasta as\n> making a piz...",None
5,assassin_legit_email_5.png,assassin_legit,0,"Re: [zzzzteana] Nothing like mama used to make\n\nFrom: ""Martin Adamson"" <martin@srv0.ems.ed.ac.uk>\n\n> | just had to jump in here as Carbonara is one of my favourites to make and\n> ask\n> what ...",None
6,assassin_legit_email_6.png,assassin_legit,0,"[zzzzteana] Playboy wants to go out with a bang\n\nFrom: ""Martin Adamson"" <martin@srv0.ems.ed.ac.uk>\n\nThe Scotsman - 22 August 2002\n\nPlayboy wants to go out with a bang\n\nAN AGEING Berlin pla...",None
7,assassin_legit_email_7.png,assassin_legit,0,Re: [zzzzteana] Nothing like mama used to make\n\nFrom: Stewart Smith <Stewart.Smith@ee.ed.ac.uk>\n\nMartin Adamson wrote: > > Isn't it just basically a mixture of beaten egg and bacon (or pancett...,None
8,assassin_legit_email_8.png,assassin_legit,0,"[zzzzteana] Meaningful sentences\n\nFrom: ""Martin Adamson"" <martin@srv0.ems.ed.ac.uk>\n\nThe Scotsman\nThu 22 Aug 2002\nMeaningful sentences\n\nTracey Lawson\n\nIf you ever wanted to look like ""on...",None
9,assassin_legit_email_9.png,assassin_legit,0,"[SAtalk] SA CGI Configurator Scripts\n\nFrom: ""NOI Administrator"" <admin@networksonline.com>\n\n| have been trying to research via SA mirrors and search engines if a canned\nscript exists giving c...",None


In [43]:
for _, row in ocr_df.head(51).iterrows():
    print(f"--- {row['file']} ({row['folder']}) ---")
    print(row["text"])
    print()


--- assassin_legit_email_0.png (assassin_legit) ---
Re: New Sequences Window

From: Robert Elz <kre@munnari.OZ.AU>

Date: Wed, 21 Aug 2002 10:54:46 -0500_ From: Chris Garrigues Message-ID:
<1029945287.4797.TMDA@deepeddy.vircio.com> || can't reproduce this error. For me it is very
repeatable... (like every time, without fail). This is the debug log of the pick happening ... 18:19:03 Pick_It
{exec pick +inbox -list -Ibrace -lbrace -subject ftp -rbrace -rbrace} {4852-4852 -sequence mercury}
18:19:03 exec pick +inbox -list -lbrace -lbrace -subject ftp -rbrace -rbrace 4852-4852 -sequence mercury
18:19:04 Ftoc_PickMsgs {{1 hit}} 18:19:04 Marking 1 hits 18:19:04 tkerror: syntax error in expression "int ...
Note, if | run the pick command by hand ... delta$ pick +inbox -list -lbrace -lbrace -subject ftp -rbrace -
rbrace 4852-4852 -sequence mercury 1 hit That's where the "1 hit" comes from (obviously). The version of
nmh I'm using is ... delta$ pick -version pick -- nmh-1.0.4 [compiled on fuchs

In [44]:
image_paths = image_paths[:51]   # add this right after the sorted(...) line


In [45]:
print(ocr_df.head(51).groupby("folder").size())

folder
assassin_legit       50
assassin_phishing     1
dtype: int64


In [46]:
subset = ocr_df.groupby("folder").head(13).reset_index(drop=True)
print(subset.groupby("folder").size())

folder
assassin_legit       13
assassin_phishing    13
trec06_legit         13
trec06_phishing      13
dtype: int64


In [47]:
import re

def clean_ocr(text):
    text = str(text)
    text = re.sub(r"-\n", "", text)                 # rejoin words split across lines ("veri-\nfy")
    text = re.sub(r"[^\w\s@:/.$%!?'-]", " ", text)  # drop junk symbols, keep email/URL/money characters
    text = re.sub(r"\s+", " ", text)                # collapse whitespace
    return text.strip().lower()

subset["clean_text"] = subset["text"].apply(clean_ocr)
subset[["file", "folder", "clean_text"]].head()

,file,folder,clean_text
0,assassin_legit_email_0.png,assassin_legit,re: new sequences window from: robert elz kre@munnari.oz.au date: wed 21 aug 2002 10:54:46 -0500_ from: chris garrigues message-id: 1029945287.4797.tmda@deepeddy.vircio.com can't reproduce this er...
1,assassin_legit_email_1.png,assassin_legit,zzzzteana re: alexander from: steve burt steve_burt@cursor-system.com martin a posted: tassos papadopoulos the greek sculptor behind the plan judged that the limestone of mount kerdylio 70 miles e...
2,assassin_legit_email_2.png,assassin_legit,zzzzteana moscow bomber from: tim chapman timc@2ubh.com man threatens explosion in moscow thursday august 22 2002 1:40 pm moscow ap - security officers on thursday seized an unidentified man who s...
3,assassin_legit_email_3.png,assassin_legit,irr klez: the virus that won't die from: monty solomon monty@roscom.com klez: the virus that won't die already the most prolific virus ever klez continues to wreak havoc. andrew brandt from the se...
4,assassin_legit_email_4.png,assassin_legit,re: zzzzteana nothing like mama used to make from: stewart smith stewart.smith@ee.ed.ac.uk in adding cream to spaghetti carbonara which has the same effect on pasta as making a pizza a deep-pie ju...


In [48]:
subset.to_csv("../data/ocr_results_clean.csv", index=False)
print("Saved cleaned OCR data")

Saved cleaned OCR data
